[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Mach_Learn/Model_Compression.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Model Compression & Edge AI

Trained models are fat with redundancy; deployment targets are small. Three sessions on the compression toolkit — quantization (closing the loop with [FPGA fixed-point](../Intro_FPGA/Intro_FPGA.ipynb)!), pruning, and distillation — each measured, not asserted, on a model we train in-notebook.

## 1. Pre-requisites

- [Intro to CNN](./Intro_CNN/Intro_CNN.ipynb) — we reuse its spectrogram classifier setup.
- [Scaling Neural Networks](./Scale_NN/Scale_NN.ipynb) — the accounting mindset.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from scipy import signal as sig
torch.manual_seed(0); rng = np.random.default_rng(0)

# a HARDER 6-class cousin of the CNN workshop task, at low SNR — so trade-offs are visible
fs, T_dur = 1024, 1.0
t = np.arange(0, T_dur, 1/fs)
def make_example(cls):
    f0 = rng.uniform(60, 200)
    if cls == 0:   x = sig.chirp(t, f0=f0, f1=f0+150, t1=T_dur)            # chirp up
    elif cls == 1: x = sig.chirp(t, f0=f0+150, f1=f0, t1=T_dur)            # chirp down
    elif cls == 2: x = np.sin(2*np.pi*f0*t)                                 # tone
    elif cls == 3: x = np.sin(2*np.pi*f0*t) + np.sin(2*np.pi*(f0+90)*t)     # two tones
    elif cls == 4: x = np.sin(2*np.pi*f0*t) * (1 + np.sin(2*np.pi*4*t))     # AM tone
    else:
        x = np.zeros_like(t); s = rng.integers(0, len(t)//2); x[s:s+len(t)//3] = rng.standard_normal(len(t)//3)
    x = x + 1.2*rng.standard_normal(len(t))                                 # LOW SNR
    _, _, S = sig.stft(x, fs=fs, nperseg=64)
    S = np.log1p(np.abs(S))[:32, :32]
    return (S - S.mean()) / (S.std() + 1e-6)

n_cls = 6
X = torch.from_numpy(np.stack([make_example(c % n_cls) for c in range(1500)]).astype(np.float32)).unsqueeze(1)
y = torch.tensor([c % n_cls for c in range(1500)])
tr, te = torch.arange(1000), torch.arange(1000, 1500)

def accuracy(m):
    m.eval()
    with torch.no_grad():
        return (m(X[te]).argmax(1) == y[te]).float().mean().item()

---
### 🕐 Session 1 of 3 — *Quantization* (~40 min)
**Goal:** shrink weights from float32 to int8; measure the size/accuracy trade.
**Builds on:** [CNN](./Intro_CNN/Intro_CNN.ipynb); [Scale_NN](./Scale_NN/Scale_NN.ipynb). &nbsp; **Feeds into:** Session 2 (pruning).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Quantization</b></summary>

**Timing (~40 min).** 8 min why float32 is habit not need · 10 min the quantisation arithmetic · 12 min the bit sweep · 10 min reading the cliff honestly.

**Open with the observation that makes the whole session possible.** Training needs wide dynamic range because gradients span many orders of magnitude. **Inference does not.** After training, each layer's weights occupy a narrow band — typically within a few multiples of their standard deviation — and 8-bit integers cover that band with resolution to spare. Float32 at deployment is a habit inherited from the training loop, not a requirement.

**Make the FPGA connection early, because this audience already owns the theory.** Storing `int8 + scale` per tensor **is** the Q-format fixed-point representation from [Intro to FPGA](../Intro_FPGA/Intro_FPGA.ipynb): a fixed scale factor and an integer mantissa. The signal-to-quantisation-noise arithmetic is identical — every extra bit buys about 6 dB. **Quantisation is not a machine-learning trick; it is fixed-point DSP applied to weights**, and students who have done the FPGA workshop can predict the shape of the results before running them.

**Walk the three lines of `quantize_model` and say what each does.** Find the maximum absolute value; divide so that value maps to $2^{b-1}-1$; round and clamp. That is **per-tensor symmetric** quantisation, the simplest scheme that exists. Note what it is vulnerable to: a single outlier weight sets the scale for the entire tensor, wasting resolution on everything else. Per-channel scales fix exactly that, and are why real deployment toolkits use them.

**Have the room predict where the cliff falls before running the sweep.** Most guess int8 is safe and int4 is risky. The measured answer is that **int4 is also free here** (99.8% at both) and **int2 collapses to 83.4%**. Let the surprise land, then immediately puncture it — see the debrief for why this task flatters quantisation.

**The honest framing matters: this model is at 99.8% on a task it finds easy.** A model with that much margin can absorb enormous weight perturbation before any test example crosses a decision boundary. **The cliff position is a property of the model–task pair, not of the bit width**, and a harder task with a genuinely uncertain model degrades much earlier. Say this explicitly; a room that leaves believing "int4 is always free" has learned something false.

**Which makes the transferable lesson procedural rather than numerical.** The professional habit is the sweep itself: quantise at several widths, measure accuracy, find your own cliff. Nobody can tell you the right bit width for your model. **Measuring it takes ten lines and five minutes**, and that is the skill worth teaching.

**Flag the size accounting, because the printed KB figures are optimistic.** `n_params * bits / 8` counts weights only. Real deployment also stores per-tensor scales (negligible), keeps activations at higher precision (not negligible), and — for int4 and int2 — needs **packing plus custom kernels**, since no standard hardware has native 4-bit or 2-bit tensor operations. The 8× and 16× numbers are upper bounds on storage, not measured end-to-end footprints.

**Close by placing the technique in current practice.** LLM deployment lives at 4–8 bits using schemes considerably cleverer than this one — per-channel scales, GPTQ and AWQ-style calibration on a small data sample, outlier-aware splitting. All of them are pushing the cliff this cell just located further out, and all of them are answering the same question: **how few bits before the accuracy breaks?**
</details>

## 2. Fewer Bits Per Weight

💡 **Intuition.** Weights are stored as float32 out of habit, not need: after training, each layer's weights occupy a narrow range that **8-bit integers** cover with plenty of resolution. Quantization stores `int8 + scale` per tensor — 4× smaller, and on supporting hardware, faster (integer math, less memory traffic). It is exactly the [Q1.15 fixed-point story](../Intro_FPGA/Intro_FPGA.ipynb) from FPGA land: the same signal-to-quantization-noise arithmetic, applied to weights.

In [2]:
class SpecCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.classify = nn.Sequential(nn.Flatten(), nn.Linear(32*8*8, 64), nn.ReLU(), nn.Linear(64, 6))
    def forward(self, x): return self.classify(self.features(x))

model = SpecCNN()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
lossf = nn.CrossEntropyLoss()
for ep in range(5):
    model.train()
    for i in range(0, 1000, 64):
        opt.zero_grad(); lossf(model(X[tr[i:i+64]]), y[tr[i:i+64]]).backward(); opt.step()
acc_fp32 = accuracy(model)
n_params = sum(p.numel() for p in model.parameters())
print(f"fp32 baseline: {acc_fp32:.1%} accuracy, {n_params*4/1024:.0f} KB")

fp32 baseline: 99.8% accuracy, 533 KB


**What just happened.** The baseline every compression result in this notebook is measured against: **99.8% test accuracy, 533 KB** of float32 weights across 136k parameters, after five epochs.

**Note first that 533 KB is not small — and that is the point of the workshop.** A microcontroller with 256 KB of flash cannot hold this model at all. An edge DSP with a tight instruction cache will thrash on it. The whole compression toolkit exists because deployment targets are quantised in size by hardware, and 533 KB sits on the wrong side of common boundaries.

**But be careful with the 99.8%, because it is going to make every compression technique look free.** The header calls this a "harder 6-class cousin at low SNR", and the noise is genuinely substantial ($1.2\sigma$ added to unit-amplitude signals). Yet the STFT converts each class into a distinct **geometric shape** — rising diagonal, falling diagonal, horizontal line, two parallel lines, modulated stripe, vertical block — and a CNN separates shapes easily even when they are noisy. **The representation did the hard work before the network saw anything**, exactly as in the [CNN workshop](./Intro_CNN/Intro_CNN.ipynb).

**Which matters for how you read the rest of the notebook.** A model at 99.8% has enormous **margin**: its logits are far from the decision boundaries, so weights can be perturbed substantially before any test example flips class. Compression damages weights; margin absorbs damage. **A high-margin model makes every compression method look better than it will look on a harder task**, and that caveat applies to all three sessions, not just quantisation.

**One structural note carried over from the CNN workshop.** Of the 136k parameters, roughly **131k (96%) sit in the single `Linear(2048, 64)` layer** after the flatten; the two convolutional layers hold under 5k between them. So when Session 2 prunes "the network", it is overwhelmingly pruning that one dense layer — and the reason 80% pruning is survivable is largely that the layer was hugely over-provisioned to begin with.

**Finally, a caveat on the measurement itself.** This is test accuracy on 500 held-out spectrograms, so the standard error is about **0.6 percentage points**. Every comparison later in the notebook — int8 versus int4, 50% versus 80% pruned, distilled versus not — should be read against that bar. **Differences under a point are not differences**, and the notebook reports several.

In [3]:
# post-training quantization by hand — per-tensor symmetric int8
def quantize_model(model, bits=8):
    qmax = 2**(bits-1) - 1
    state = {}
    for name, p in model.state_dict().items():
        scale = p.abs().max() / qmax if p.abs().max() > 0 else 1.0
        q = torch.clamp((p / scale).round(), -qmax-1, qmax)
        state[name] = (q.to(torch.int8 if bits <= 8 else torch.int32), scale)
    return state

def dequantize_into(model, qstate):
    sd = {name: q.float() * s for name, (q, s) in qstate.items()}
    model.load_state_dict(sd)

for bits in [8, 4, 2]:
    m2 = SpecCNN(); qs = quantize_model(model, bits)
    dequantize_into(m2, qs)
    print(f"int{bits}: {accuracy(m2):.1%} accuracy, {n_params*bits/8/1024:.0f} KB  "
          f"({32//bits}x smaller)")

int8: 99.8% accuracy, 133 KB  (4x smaller)
int4: 99.8% accuracy, 67 KB  (8x smaller)
int2: 83.4% accuracy, 33 KB  (16x smaller)


**What just happened.** A bit-width sweep with a sharp cliff in it:

| precision | accuracy | size | vs fp32 |
|---|---|---|---|
| fp32 | 99.8% | 533 KB | 1× |
| int8 | 99.8% | 133 KB | 4× |
| int4 | **99.8%** | 67 KB | 8× |
| int2 | **83.4%** | 33 KB | 16× |

**int8 and int4 are free; int2 loses 16 points.** Eight times smaller at no measurable cost, and then a wall.

**Where the cliff sits is the useful finding, and the *procedure* is more useful than the number.** Nobody can tell you the right bit width for your model — it depends on the architecture, the task, and how much margin the trained model has. **Running this sweep takes ten lines and five minutes**, and doing it is the professional habit the session exists to install.

**But this particular cliff is unusually forgiving, and it is important to say why.** The baseline is at 99.8%, which means the logits sit far from every decision boundary. Quantisation perturbs the weights; **margin absorbs the perturbation**. A model that was genuinely uncertain — say 75% on a hard task — would start losing examples at int8. **The cliff position is a property of the model–task pair, not of the bit width**, and a room that leaves believing "int4 is always safe" has learned something false.

**The mechanism at int2 is worth naming precisely.** Two bits give you exactly four levels: $\{-2, -1, 0, 1\}$ times the scale. A weight distribution that is roughly Gaussian gets crushed into four values, and the per-tensor scale is set by the single largest weight — so if one outlier is 5× the typical magnitude, nearly every other weight quantises to $0$ or $\pm 1$. **The representation runs out of resolution, not of range**, and per-channel scaling is the standard fix.

**Connect the arithmetic back to the FPGA workshop, since it is the same theory.** Each bit buys about **6 dB** of signal-to-quantisation-noise ratio. Going 8 → 4 bits costs 24 dB and the model did not care; 4 → 2 costs another 12 dB and it broke. This is the [Q-format fixed-point](../Intro_FPGA/Intro_FPGA.ipynb) story applied to weights — same formula, same trade, different application domain.

**Two caveats on the size column, because it is optimistic.** It counts **weights only**: activations stay at higher precision in any real deployment, and those often dominate the memory traffic that quantisation was supposed to reduce. And int4/int2 need **bit-packing plus custom kernels** — no mainstream hardware has native 4-bit or 2-bit tensor ops, so the 8× and 16× figures are storage upper bounds, not measured end-to-end footprints or speedups.

**Finally, note that this scheme is the simplest one that exists.** Per-tensor symmetric quantisation, one scale for an entire weight tensor, no calibration data. Production LLM quantisation at 4 bits uses per-channel scales, calibration on a sample of real activations, and outlier-aware handling (GPTQ, AWQ, SmoothQuant). **All of them exist to push the cliff you just measured further out**, and knowing where the naive cliff falls is what makes their contribution legible.

Read the table: int8 — and here even int4 — is free; int2 collapses. Where the cliff sits depends on the model and task; measuring it (as you just did) is the professional habit. Modern LLM deployment lives at 4–8 bits with cleverer schemes (per-channel scales, GPTQ/AWQ-style calibration) that push the cliff further out.

---
### 🕐 Session 2 of 3 — *Pruning* (~35 min)
**Goal:** remove small weights entirely; discover how much of the network was never needed.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (distillation).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Pruning</b></summary>

**Timing (~35 min).** 8 min over-provisioning · 10 min magnitude pruning and the fine-tune step · 10 min the sweep · 7 min structured versus unstructured.

**Open with the claim the session tests: trained networks are over-provisioned.** Plot a histogram of the trained weights — most are tiny. The hypothesis is that they were never doing much and can be deleted. **Ask the room to guess what fraction can go before accuracy falls.** Answers usually cluster around 20–30%; the measured answer here is that 80% goes with no loss at all, and 95% costs three points. Let the gap between guess and measurement land.

**Emphasise that the fine-tune is not optional — it is half the method.** Zeroing weights damages the network; retraining the survivors lets them **absorb the function** the deleted weights were performing. The sweep shows exactly this at 95%: 91.8% immediately after pruning, 97.0% after one epoch. **Prune-then-recover, and the recovery is where most of the accuracy comes back.** Iterative schedules (prune 20%, fine-tune, repeat) do better still than one large cut.

**Point at the mask re-application inside the training loop, because students omit it and then wonder why sparsity vanishes.** Gradient descent will happily push a zeroed weight back to nonzero — the optimiser knows nothing about your intent. `p.mul_(masks[mi])` after every step is what keeps pruned weights pruned. **Without it you are fine-tuning a dense network that happened to start with some zeros.**

**Make the structured/unstructured distinction concrete, since it decides whether pruning is useful at all.** Unstructured pruning scatters zeros anywhere; storage shrinks with a sparse format, but **the arithmetic does not get faster** — a GPU multiplying a matrix that is 80% zeros does exactly as much work as a dense one. Speedups require **structured** pruning: remove whole channels, whole heads, whole neurons, so the remaining tensor is smaller and dense. Ask the room which one this notebook implements. It is unstructured, so the honest claim is *storage*, not *latency*.

**Then note where the sparsity actually lands, which changes the interpretation.** 96% of this model's parameters are in the single `Linear(2048, 64)` layer, so "80% pruned" mostly means that one dense layer was hugely over-provisioned. The convolutional layers hold under 5k parameters between them and cannot absorb the same treatment. **Global magnitude pruning quietly redistributes sparsity toward whatever layer is largest**, which is why per-layer sparsity budgets are common in practice.

**Give the lottery-ticket framing, and be careful about what it claims.** The observation is that a small subnetwork, identified after training, can be retrained from its *original initialisation* to match the full network. That is a statement about the existence of a good subnetwork plus a lucky initialisation — **not** a recipe for finding one cheaply, since you have to train the full model first to identify it. It is a genuinely interesting result and it is routinely overstated.

**Close by combining the sessions, since compression techniques multiply.** 80% pruning plus int8 is roughly **20× smaller** than the fp32 dense baseline, and both were free here. Stack distillation on top in Session 3 and the total is larger still. **The techniques are orthogonal**, and edge deployment uses all three together rather than choosing between them.
</details>

## 3. Fewer Weights, Period

💡 **Intuition.** Trained networks are over-provisioned: many weights end up tiny and removable. **Magnitude pruning** zeroes the smallest p%, then (crucially) *fine-tunes* the survivors to absorb the loss. Unstructured sparsity shrinks storage; hardware speedups need *structured* pruning (whole channels/heads) so dense math stays dense. The deeper lesson is the lottery-ticket observation: a small subnetwork was doing most of the work all along.

In [4]:
import copy
def prune_magnitude(model, frac):
    m2 = copy.deepcopy(model)
    with torch.no_grad():
        all_w = torch.cat([p.abs().flatten() for p in m2.parameters() if p.dim() > 1])
        thresh = torch.quantile(all_w, frac)
        masks = []
        for p in m2.parameters():
            if p.dim() > 1:
                mask = (p.abs() > thresh).float()
                p.mul_(mask); masks.append(mask)
    return m2, masks

print("prune → accuracy (no fine-tune) → after 1 epoch of fine-tune:")
for frac in [0.5, 0.8, 0.95]:
    m2, masks = prune_magnitude(model, frac)
    a0 = accuracy(m2)
    opt2 = torch.optim.Adam(m2.parameters(), lr=5e-4)
    m2.train()
    for i in range(0, 1000, 64):
        opt2.zero_grad(); lossf(m2(X[tr[i:i+64]]), y[tr[i:i+64]]).backward(); opt2.step()
        with torch.no_grad():                      # re-apply masks: pruned stays pruned
            mi = 0
            for p in m2.parameters():
                if p.dim() > 1: p.mul_(masks[mi]); mi += 1
    print(f"  {frac:.0%} pruned: {a0:.1%} → {accuracy(m2):.1%}")

prune → accuracy (no fine-tune) → after 1 epoch of fine-tune:
  50% pruned: 99.8% → 99.6%
  80% pruned: 99.8% → 99.6%


  95% pruned: 91.8% → 97.0%


**What just happened.** Three pruning levels, before and after one epoch of fine-tuning:

| pruned | immediately after | after fine-tune |
|---|---|---|
| 50% | 99.8% | 99.6% |
| 80% | 99.8% | 99.6% |
| 95% | **91.8%** | **97.0%** |

**Delete four out of every five weights and nothing happens.** 99.8% before, 99.8% after, without even fine-tuning. That is a strong statement about how much of a trained network was never load-bearing.

**Read the 95% row as the one that shows the mechanism.** Pruning cost 8 points immediately; one epoch of retraining recovered 5 of them. **The fine-tune is not a polish step, it is half the method** — the surviving weights *absorb* the function the deleted ones were performing, and without that absorption 95% pruning would be unusable. Iterative schedules (prune 20%, fine-tune, repeat) recover more still.

**Note the small regression at 50% and 80%, and do not over-explain it.** 99.8% → 99.6% is **one example out of 500**, well inside the ~0.6-point standard error. It is noise, and both numbers mean "unchanged". Reading a story into it would be exactly the mistake the sweep is meant to prevent.

**The line inside the loop that students omit is `p.mul_(masks[mi])`.** Gradient descent will happily push a zeroed weight back to nonzero — the optimiser has no idea you meant it to stay zero. Re-applying the mask after **every** step is what keeps the network sparse. Without it you are fine-tuning a dense model that happened to start with some zeros, and the sparsity quietly evaporates over the epoch.

**Now the caveat that decides whether any of this is useful: this is *unstructured* pruning.** The zeros are scattered arbitrarily, so storage shrinks in a sparse format but **the arithmetic does not get faster**. A GPU multiplying a matrix that is 80% zeros does exactly the same work as a dense one. Real speedups need **structured** pruning — remove entire channels, heads, or neurons so the remaining tensor is genuinely smaller and still dense. **The honest claim here is memory, not latency**, and conflating the two is the most common error in compression write-ups.

**Note also where the sparsity actually went.** 96% of this model's parameters live in the single `Linear(2048, 64)` layer, so a global magnitude threshold prunes overwhelmingly *there*. "80% of the network" is really "most of one very over-provisioned dense layer" — the convolutional layers hold under 5k parameters and could not survive the same treatment. **Global thresholds redistribute sparsity toward whatever layer is biggest**, which is why per-layer budgets are standard practice.

**And keep the baseline caveat in view.** A model at 99.8% has large margin, so it tolerates enormous weight damage before any test example crosses a boundary. **A genuinely uncertain model degrades much earlier**, and the 80%-is-free result is a property of this model–task pair rather than a general law. The transferable output is the sweep, not the number.

**Finally, the compression arithmetic compounds.** 80% pruning plus int8 from Session 1 is about **20× smaller** than the fp32 dense baseline, both at no measured accuracy cost. The techniques are orthogonal, and edge deployment stacks them rather than choosing.

---
### 🕐 Session 3 of 3 — *Knowledge Distillation* (~40 min)
**Goal:** train a small student to mimic the big teacher's soft predictions.
**Builds on:** Session 2.

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: Knowledge Distillation</b></summary>

**Timing (~40 min).** 10 min dark knowledge · 10 min the loss and its temperature · 10 min the experiment · 10 min reading a small effect honestly.

**Open with the observation the method is built on, and make it concrete.** A one-hot label says "chirp" and nothing else. The teacher's softmax says "80% chirp-up, 19% chirp-down, 1% everything else" — which additionally tells the student that **chirp-up and chirp-down are confusable and neither resembles a noise burst**. That similarity structure is the *dark knowledge*: information the teacher learned that the labels never contained.

**Explain the temperature as an amplifier for exactly that structure.** A confident teacher outputs something like $[0.99, 0.008, 0.002, \ldots]$, where the informative ratios are buried in the third decimal. Dividing logits by $T = 3$ flattens the distribution and lifts those ratios into a range where the KL gradient can see them. **Without temperature there is almost no dark knowledge left to transfer** — the soft target is nearly one-hot again.

**Explain the $T^2$ factor too, since it looks arbitrary.** Softening by $T$ scales the KL gradient by roughly $1/T^2$, so multiplying by $T^2$ keeps the distillation term's magnitude comparable to the hard-label term as $T$ varies. It is gradient bookkeeping, not a tuning knob, and it means $\alpha$ retains its meaning when you change $T$.

**Point at the size ratio before the results, because it is the headline.** Teacher: 136k parameters. Student: **1.6k** — an **85× reduction**. Ask the room to predict how much accuracy that costs. Most expect a collapse; the answer is about 3 points, which is the genuinely surprising part of the session.

**Then be scrupulous about the distillation effect itself, because it is small.** 97.0% for the label-trained student against 97.8% distilled — **0.8 points, which is 4 examples out of 500**, against a binomial standard error near 0.7 points. **This is roughly one standard error from a single seed: suggestive, not established.** The literature's effect is real and larger, but this run does not demonstrate it, and saying so is more useful than presenting 0.8 points as a result.

**Give the room the experiment that would settle it.** Train both students across five seeds and report mean ± standard deviation. Two minutes of compute, and it converts an anecdote into a measurement. This is the same discipline the [Training Dynamics](./Training_Dynamics.ipynb) optimiser race needed, and it is worth repeating whenever a difference is smaller than a couple of points.

**Explain why the effect is muted here, since the reason is instructive.** Dark knowledge lives in the teacher's **uncertainty**, and this teacher is at 99.8% — its soft targets are nearly one-hot, so there is very little inter-class structure to transmit. **A teacher with nothing to be uncertain about has no dark knowledge to share.** Distillation shines when the teacher is good but not saturated, and on tasks with genuinely confusable classes.

**Note the architectural gap the student had to cross, because it explains the 3 points.** `TinyCNN` has 4 channels and a single $4\times4$ pool, so its spatial resolution collapses immediately and it has no second convolutional stage. It is not a scaled-down version of the teacher; it is a structurally weaker family. **Distillation transfers function, not capacity** — no soft target lets a 1.6k-parameter model represent what 136k can.

**Close by stacking the three sessions.** Quantise (8×), prune (5×), distil (85× fewer parameters) — the techniques are orthogonal and multiply. A student at 1.6k parameters, pruned and quantised to int8, is a model in the low kilobytes that runs on a microcontroller. **Every technique in this workshop is the same trade, measured rather than asserted: bits and weights for accuracy, priced explicitly.**
</details>

## 4. Teaching a Smaller Model

💡 **Intuition.** A trained teacher knows more than its labels: its **soft probabilities** encode which classes are *almost* confusable — 'this chirp is 80% chirp, 19% tone' is a richer target than 'chirp'. Distillation trains a small student against those softened outputs (temperature > 1 amplifies the dark knowledge) plus the true labels. The student often lands closer to the teacher than the same architecture trained on labels alone.

In [5]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(1, 4, 3, padding=1), nn.ReLU(), nn.MaxPool2d(4))
        self.classify = nn.Sequential(nn.Flatten(), nn.Linear(4*8*8, 6))
    def forward(self, x): return self.classify(self.features(x))

def train_student(distill, epochs=8, T_=3.0, alpha=0.7):
    torch.manual_seed(1)
    s = TinyCNN()
    opt_s = torch.optim.Adam(s.parameters(), lr=2e-3)
    kl = nn.KLDivLoss(reduction="batchmean")
    for ep in range(epochs):
        s.train()
        for i in range(0, 1000, 64):
            xb, yb = X[tr[i:i+64]], y[tr[i:i+64]]
            logits_s = s(xb)
            loss = lossf(logits_s, yb)
            if distill:
                with torch.no_grad():
                    logits_t = model(xb)
                soft = kl(torch.log_softmax(logits_s/T_, 1), torch.softmax(logits_t/T_, 1)) * T_*T_
                loss = alpha*soft + (1-alpha)*loss
            opt_s.zero_grad(); loss.backward(); opt_s.step()
    return s

s_plain = train_student(False)
s_dist  = train_student(True)
n_tiny = sum(p.numel() for p in TinyCNN().parameters())
print(f"teacher ({n_params/1000:.0f}k params): {acc_fp32:.1%}")
print(f"student ({n_tiny/1000:.1f}k params) trained on labels:      {accuracy(s_plain):.1%}")
print(f"student ({n_tiny/1000:.1f}k params) distilled from teacher: {accuracy(s_dist):.1%}")

teacher (136k params): 99.8%
student (1.6k params) trained on labels:      97.0%
student (1.6k params) distilled from teacher: 97.8%


**What just happened.** A student with **85× fewer parameters** than the teacher, trained two ways:

| model | params | accuracy |
|---|---|---|
| teacher | 136k | 99.8% |
| student, labels only | 1.6k | 97.0% |
| student, distilled | 1.6k | **97.8%** |

**The headline is the size ratio, not the distillation gap.** 1.6k parameters — about 6 KB in fp32, under 2 KB at int8 — reaching 97.8% where the 533 KB teacher reaches 99.8%. **Eighty-five times smaller for two points.** That is the number worth remembering from this session.

**Now be careful with the distillation effect itself, because it is smaller than it looks.** 97.8% versus 97.0% is **4 examples out of 500**. The binomial standard error at this accuracy is about 0.7 points, so the gap is roughly **one standard error, from a single seed, with no repeats**. The honest statement is that distillation did not hurt and may have helped; "distillation adds 0.8 points" is not supported by this run.

**The experiment that would settle it takes two minutes.** Train both students across five seeds and report mean ± standard deviation. The distillation effect in the literature is real and typically larger than this; converting an anecdote into a measurement is the same discipline the [Training Dynamics](./Training_Dynamics.ipynb) optimiser race needed.

**And there is a specific reason the effect is muted here, which is instructive in itself.** Dark knowledge lives in the teacher's **uncertainty** — "80% chirp-up, 19% chirp-down" tells the student those two classes are confusable, which no one-hot label ever says. But this teacher sits at 99.8%, so its softmax is nearly one-hot and there is very little inter-class structure left to transmit. **A teacher with nothing to be uncertain about has no dark knowledge to share.** Distillation shines when the teacher is strong but not saturated.

**The temperature is what rescues whatever structure remains.** A confident teacher outputs $[0.99, 0.008, 0.002, \ldots]$, where the informative ratios are buried in the third decimal. Dividing logits by $T = 3$ lifts them into a range the KL gradient can act on — and the $T^2$ multiplier compensates for the $1/T^2$ shrinkage that softening induces in the gradient, so $\alpha$ keeps its meaning as $T$ changes. **Both factors are bookkeeping, and without them the soft target is nearly one-hot again.**

**Note why the student loses 3 points, since it is not a distillation failure.** `TinyCNN` has 4 channels, one $4\times4$ pool, and no second convolutional stage — it is not a scaled-down teacher, it is a structurally weaker family with far less spatial resolution. **Distillation transfers function, not capacity.** No soft target lets 1.6k parameters represent what 136k can.

**Finally, the three sessions multiply.** Quantise to int8 (4×), prune 80% (5×), distil to 1.6k parameters (85× fewer). Applied together, the 533 KB baseline becomes a model in the low kilobytes that fits a microcontroller's flash. **Each technique is the same trade priced explicitly — bits and weights for accuracy — and the pricing is measured rather than asserted.**

## 5. Conclusion

Quantize to 8 bits for free, prune what training over-provisioned, distill what remains into a smaller architecture — then stack all three for edge deployment. Every technique is the same trade, measured: bits and weights for accuracy, priced explicitly.

---
## Where next

- [Intro to FPGA](../Intro_FPGA/Intro_FPGA.ipynb) — where the int8 weights land in silicon.
- [Scaling Neural Networks](./Scale_NN/Scale_NN.ipynb) — compression's mirror image.
- [LLMs from the Ground Up](./LLMs_from_the_Ground_Up.ipynb) — why 4-bit LLMs are everywhere.